In [ ]:
from diffusers.utils import load_image, make_image_grid
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline

import torch
import pathlib

from IPython.display import display

In [ ]:
root_path = pathlib.Path("/home/leafying/data/UAV/DUT_Anti_UAV/detection/images/val")
image_paths = sorted(root_path.rglob("*.jpg"))[:5]

init_images = [load_image(image_path.as_posix()) for image_path in image_paths]
for init_image in init_images:
    print(init_image.size)
make_image_grid(init_images, rows=1, cols=len(init_images), resize=512)

In [ ]:
base_text2img = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

base_text2img.enable_model_cpu_offload()

## Image to Image

In [ ]:
base = StableDiffusionXLImg2ImgPipeline(**base_text2img.components)
refiner = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    text_encoder_2=base.text_encoder_2,
    vae=base.vae,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

refiner.enable_model_cpu_offload()

### Base

In [ ]:
generator = torch.Generator().manual_seed(0)
prompt = "drone flying in foggy weather"

In [ ]:
base_image = base_text2img(prompt, generator=generator).images[0]
display(base_image)

In [ ]:
base_images = []
for init_image in init_images:
    base_image = base(prompt, image=init_image, generator=generator).images[0]
    display(base_image)
    base_images.append(base_image)
# make_image_grid(base_images, rows=1, cols=len(base_images))

### Base + Refiner

In [ ]:
for init_image, base_image in zip(init_images, base_images):
    refined_image = base(
        prompt,
        image=init_image,
        generator=generator,
        output_type="latent",
        num_inference_steps=40,
        denoising_end=0.8,
    ).images
    refined_image = refiner(
        prompt,
        image=refined_image,
        generator=generator,
        num_inference_steps=40,
        denoising_start=0.8,
    ).images[0]
    display(make_image_grid([init_image, base_image, refined_image], rows=1, cols=3))

In [ ]:
base = StableDiffusionXLImg2ImgPipeline(**base_text2img.components)
refiner = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    text_encoder_2=base.text_encoder_2,
    vae=base.vae,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

refiner.enable_model_cpu_offload()